# 🏥 Healthcare Cost Prediction

## Objective
Predict insurance charges using demographic and health features.

## Business Impact
Helps insurance companies in pricing policies and identifying high-risk individuals.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('insurance.csv')
df.head()

In [ ]:
X = df.drop('charges', axis=1)
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
num_features = ["age", "bmi", "children"]
cat_features = ["sex", "smoker", "region"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder()),
    ("scaler", StandardScaler(with_mean=False))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "Linear Regression": LinearRegression(),
    "XGBRegressor": XGBRegressor(),
    "AdaBoost": AdaBoostRegressor()
}

params = {
    "Random Forest": {"model__n_estimators": [8, 16, 32, 64]},
    "Gradient Boosting": {"model__learning_rate": [0.1, 0.01], "model__n_estimators": [32, 64]},
    "XGBRegressor": {"model__learning_rate": [0.1, 0.01], "model__n_estimators": [32, 64]},
    "AdaBoost": {"model__n_estimators": [32, 64]},
    "Decision Tree": {},
    "Linear Regression": {}
}

In [ ]:
results = {}
best_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    param = params.get(name, {})

    if param:
        grid = GridSearchCV(pipe, param, cv=3)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_
    else:
        best_model = pipe.fit(X_train, y_train)

    y_pred = best_model.predict(X_test)
    score = r2_score(y_test, y_pred)

    results[name] = score
    best_models[name] = best_model

results

In [ ]:
best_model_name = max(results, key=results.get)
best_model = best_models[best_model_name]

print("Best Model:", best_model_name)
print("R2 Score:", results[best_model_name])

In [ ]:
model = best_model.named_steps['model']
preprocessor_fitted = best_model.named_steps['preprocessor']

ohe = preprocessor_fitted.named_transformers_['cat']
cat_names = ohe.named_steps['onehot'].get_feature_names_out(cat_features)

all_features = num_features + list(cat_names)

importances = model.feature_importances_

fi_df = pd.DataFrame({
    "Feature": all_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

fi_df.head()

In [ ]:
plt.figure()
plt.barh(fi_df['Feature'], fi_df['Importance'])
plt.gca().invert_yaxis()
plt.title("Feature Importance")
plt.show()

### Conclusion
- Smoking is the most influential factor affecting medical cost
- BMI and age are key contributors
- Model helps in insurance pricing and risk segmentation